# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Kojo Kootin-Sanwu]
**Student ID:** [63952028]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [5]:
# API-key setup — DO NOT hard-code your key in this cell.

import os


# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)


API_KEY = os.environ["GROQ_API_KEY"]

client = OpenAI(api_key=API_KEY, base_url="https://api.groq.com/openai/v1")

MODEL = "llama-3.3-70b-versatile"  # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [6]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500,
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    print("Token usage:", response.usage)

    return response.choices[0].message.content


# TODO: Call it once with a simple question and print the answer.
prompt = "Save my GPA"
answer = ask_llm(
    user_prompt=prompt,
    system_prompt="You are an academic advisor",
    temperature=0.7,
    max_tokens=200,
)
# TODO: Print response.usage as well — how many tokens did your call consume?
print(answer)

Token usage: CompletionUsage(completion_tokens=200, prompt_tokens=43, total_tokens=243, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.051299667, prompt_time=0.002195466, completion_time=0.622477428, total_time=0.624672894)
Don't worry, I'm here to help. To save your GPA, we'll need to assess your current situation and create a plan to get you back on track. Please provide me with some information:

1. What is your current GPA?
2. What is your desired GPA?
3. How many credits have you completed so far?
4. What are the grades you've received in your previous semesters (if any)?
5. Are there any specific courses or subjects where you're struggling?
6. What are your academic goals (e.g., graduating with honors, getting into a specific program, etc.)?

Additionally, let's consider the following general strategies to help improve your GPA:

* **Focus on your strengths**: Identify the subjects where you excel and prioritize those courses.
* **Seek help when needed**

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** [Double-click to edit]

### Part 1.2 — Temperature: the randomness dial

In [ ]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.

prompt = "Suggest a name for a savings product for market traders in Accra."

print("TEMPERATURE = 0.0")

for i in range(5):
    answer = ask_llm(
        user_prompt=prompt,
        temperature=0.0
    )
    print(f"{i + 1}. {answer}\n")


print("\nTEMPERATURE = 1.2")
print("-----------------")

for i in range(5):
    answer = ask_llm(
        user_prompt=prompt,
        temperature=1.2
    )
    print(f"{i + 1}. {answer}\n")

TEMPERATURE = 0.0
-----------------
Token usage: CompletionUsage(completion_tokens=256, prompt_tokens=56, total_tokens=312, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.052363669, prompt_time=0.001896009, completion_time=0.685767343, total_time=0.687663352)
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders.
5. **Sika Kurom**: "Sika" means "money" in Ghanaian, and "Kurom" means "box" or "container", so this name suggests a safe and secure place to store savings.
6. **Tr

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** [Double-click to edit]

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [8]:
LETTERS = {
    "L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",
    "L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",
    "L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",
    "L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",
    "L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",
    "L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
    "L001": {
        "applicant_name": "Akosua Mensah",
        "amount_ghs": 8000,
        "purpose": "buy deep freezer / expand into frozen foods",
        "monthly_profit_ghs": 900,
        "has_collateral_or_guarantor": True,
        "repayment_months": 20,
    },
    "L003": {
        "applicant_name": "Efua Darko",
        "amount_ghs": 15000,
        "purpose": "industrial sewing machines and fabric stock",
        "monthly_profit_ghs": 2800,
        "has_collateral_or_guarantor": True,
        "repayment_months": 15,
    },
    "L006": {
        "applicant_name": "Kofi",
        "amount_ghs": 50000,
        "purpose": "car wash, provision shop, phone imports",
        "monthly_profit_ghs": None,
        "has_collateral_or_guarantor": False,
        "repayment_months": 12,
    },
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [9]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this:"

for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]

    answer = ask_llm(
        user_prompt=f"{SUMMARY_PROMPT_V1}\n\n{letter_text}",
        temperature=0
    )

    print(f"{letter_id} - V1")
    print(answer)
    print()
# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
SUMMARY_SYSTEM_PROMPT_V2 = """
You are an assistant to a microfinance loan officer.

Summarize the applicant's loan application into a concise factual brief.

Requirements:
- Write only 3-4 sentences.
- Remain factual and neutral.
- Include the most important financial and loan-related information.
- Do not invent or assume any information that is not stated in the application.
- Do not make the loan approval decision.
"""

for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]

    answer = ask_llm(
        user_prompt=f"Summarize this loan application:\n\n{letter_text}",
        system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
        temperature=0
    )

    print(f"{letter_id} - V2")
    print(answer)
    print()

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

Token usage: CompletionUsage(completion_tokens=63, prompt_tokens=133, total_tokens=196, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.05175989, prompt_time=0.00644801, completion_time=0.201851938, total_time=0.208299948)
L002 - V1
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when his finances recover.

Token usage: CompletionUsage(completion_tokens=93, prompt_tokens=135, total_tokens=228, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.052195703, prompt_time=0.00644434, completion_time=0.283169687, total_time=0.289614027)
L006 - V1
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but c

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** [Double-click to edit]

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [12]:
import json
import pandas as pd
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
EXTRACT_PROMPT = """
You are extracting structured information from a loan application letter.

Return ONLY a valid JSON object with EXACTLY the following keys:
{{"applicant_name": string,
  "amount_ghs": number,
  "purpose": string,
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": boolean,
  "repayment_months": number or null}}

Use only information explicitly stated in the letter. If a field is not stated in the letter, use null. Do not guess.
Has_collateral_or_guarantor should be true if either collateral or a guarantor is explicitly mentioned.

Example:

Letter:
"My name is Ama Boateng. I am requesting GHS 8,000 to purchase a new sewing machine.
My tailoring business earns a monthly profit of GHS 2,000. My sister has agreed to act
as my guarantor. I would like to repay the loan over 10 months."

Output:
{{"applicant_name": "Ama Boateng",
  "amount_ghs": 8000,
  "purpose": "purchase a new sewing machine",
  "monthly_profit_ghs": 2000,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10}}

Now extract the information from this letter:
{letter}
"""
# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

def extract_fields(letter_text):

    prompt = EXTRACT_PROMPT.format(letter=letter_text)

    response = ask_llm(
        user_prompt=prompt,
        temperature=0
    )

    # Remove possible markdown fences
    cleaned = response.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]

    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]

    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]

    cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)

    except json.JSONDecodeError:
        print("Warning: Could not parse model response as JSON.")
        print(response)
        return None


# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

results = []

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is not None:
        extracted["letter_id"] = letter_id
        results.append(extracted)

df_extracted = pd.DataFrame(results)

df_extracted

Token usage: CompletionUsage(completion_tokens=70, prompt_tokens=437, total_tokens=507, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.051803753, prompt_time=0.049850342, completion_time=0.122257307, total_time=0.172107649)
Token usage: CompletionUsage(completion_tokens=69, prompt_tokens=397, total_tokens=466, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.051982613, prompt_time=0.045099502, completion_time=0.119097035, total_time=0.164196537)
Token usage: CompletionUsage(completion_tokens=74, prompt_tokens=451, total_tokens=525, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.050707118, prompt_time=0.023258023, completion_time=0.123196274, total_time=0.146454297)
Token usage: CompletionUsage(completion_tokens=67, prompt_tokens=417, total_tokens=484, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.093861475, prompt_time=0.02035489, completion_time=0.123187232, total_time=0.143542122)
T

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months,letter_id
0,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0,L001
1,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN,L002
2,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0,L003
3,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0,L004
4,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0,L005
5,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0,L006


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** [Double-click to edit]

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [13]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_PROMPT = """
Read the loan application and the extracted information below.

Give me a short decision-support brief for the loan officer.

Include:

1. Strengths
- Mention the good points in the application.
- Only use information that is actually in the letter.

2. Risks / red flags
- Mention anything that could make the loan risky.

3. Missing information
- Mention important information that the loan officer may still need to ask for.

4. Suggested next step
- Suggest what the loan officer should do next, for example:
  "invite for interview",
  "request documents",
  or "flag for senior review".

Do NOT say approve or reject.
The final decision must be made by a human loan officer.

Loan application:
{letter}

Extracted information:
{extracted_json}
"""

def generate_brief(letter_text, extracted_data):

    prompt = BRIEF_PROMPT.format(
        letter=letter_text,
        extracted_json=json.dumps(extracted_data, indent=2)
    )

    brief = ask_llm(
        user_prompt=prompt,
        temperature=0
    )

    return brief

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
briefs = {}

for letter_id, letter_text in LETTERS.items():

    extracted_data = extract_fields(letter_text)

    if extracted_data is not None:
        brief = generate_brief(letter_text, extracted_data)
        briefs[letter_id] = brief

for letter_id in ["L001", "L002", "L006"]:
    print(letter_id)
    print(briefs[letter_id])

Token usage: CompletionUsage(completion_tokens=70, prompt_tokens=437, total_tokens=507, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.040879772, prompt_time=0.027857143, completion_time=0.114502377, total_time=0.14235952)
Token usage: CompletionUsage(completion_tokens=276, prompt_tokens=393, total_tokens=669, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.040809329, prompt_time=0.047015411, completion_time=0.940437532, total_time=0.987452943)
Token usage: CompletionUsage(completion_tokens=69, prompt_tokens=397, total_tokens=466, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.04181775, prompt_time=0.040924625, completion_time=0.119540129, total_time=0.160464754)
Token usage: CompletionUsage(completion_tokens=263, prompt_tokens=352, total_tokens=615, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.04069172, prompt_time=0.018371382, completion_time=0.747395553, total_time=0.765766935)
T

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** [Double-click to edit]

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 73edb96568ea47c0b4f508389f5b35c0594912e1

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [14]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

letter_ids = ["L001", "L003", "L006"]

evaluation = []

for field in fields:
    row = {"field": field}
    correct_count = 0

    for letter_id in letter_ids:

        predicted = df_extracted.loc[
            df_extracted["letter_id"] == letter_id,
            field
        ].iloc[0]

        actual = GOLD[letter_id][field]

        # Compare names without worrying about capital letters
        if field == "applicant_name":
            is_correct = str(predicted).lower() == str(actual).lower()

        else:
            is_correct = predicted == actual

        row[letter_id] = is_correct

        if is_correct:
            correct_count += 1

    row["accuracy"] = correct_count / len(letter_ids)

    evaluation.append(row)

accuracy_df = pd.DataFrame(evaluation)

accuracy_df
# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

,field,L001,L003,L006,accuracy
0,applicant_name,True,True,True,1.000000
1,amount_ghs,True,True,True,1.000000
2,purpose,False,False,False,0.000000
3,monthly_profit_ghs,True,True,False,0.666667
4,has_collateral_or_guarantor,True,True,True,1.000000
5,repayment_months,True,True,True,1.000000


### Part 4.2 — Reliability: is the system consistent?

In [15]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

def extract_fields(letter_text, temperature=0):

    prompt = EXTRACT_PROMPT.format(letter=letter_text)

    response = ask_llm(
        user_prompt=prompt,
        temperature=temperature
    )

    cleaned = response.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]

    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]

    cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)

    except json.JSONDecodeError:
        print("Warning: Could not parse model response as JSON.")
        return None


results_temp_0 = []
results_temp_1 = []

for i in range(5):
    result = extract_fields(LETTERS["L004"], temperature=0)
    results_temp_0.append(result)

for i in range(5):
    result = extract_fields(LETTERS["L004"], temperature=1.0)
    results_temp_1.append(result)


valid_temp_0 = sum(result is not None for result in results_temp_0)
valid_temp_1 = sum(result is not None for result in results_temp_1)

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

unique_temp_0 = set(
    json.dumps(result, sort_keys=True)
    for result in results_temp_0
    if result is not None
)

unique_temp_1 = set(
    json.dumps(result, sort_keys=True)
    for result in results_temp_1
    if result is not None
)


print("Temperature 0")
print("Valid JSON:", valid_temp_0, "/ 5")
print("Unique outputs:", len(unique_temp_0))
print("Identical across all valid runs:", len(unique_temp_0) == 1)

print()

print("Temperature 1.0")
print("Valid JSON:", valid_temp_1, "/ 5")
print("Unique outputs:", len(unique_temp_1))
print("Identical across all valid runs:", len(unique_temp_1) == 1)

Token usage: CompletionUsage(completion_tokens=67, prompt_tokens=417, total_tokens=484, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.040593679, prompt_time=0.02043841, completion_time=0.120979036, total_time=0.141417446)
Token usage: CompletionUsage(completion_tokens=67, prompt_tokens=417, total_tokens=484, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.041299509, prompt_time=0.034510138, completion_time=0.122042185, total_time=0.156552323)
Token usage: CompletionUsage(completion_tokens=67, prompt_tokens=417, total_tokens=484, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.040657415, prompt_time=0.02142854, completion_time=0.124582857, total_time=0.146011397)
Token usage: CompletionUsage(completion_tokens=67, prompt_tokens=417, total_tokens=484, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.040630874, prompt_time=0.021795614, completion_time=0.126633714, total_time=0.148429328)
To

### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.